In [12]:
import pandas as pd
import re
from datetime import datetime

# Read both Excel files
df1 = pd.read_excel('graduates.xlsx')
df2 = pd.read_excel('table.xlsx')

# Clean data
df1 = df1.fillna('')
df2 = df2.fillna('')

def clean_name_for_comparison(name):
    # Remove parentheses and their contents
    name = re.sub(r'\([^)]*\)', '', name)
    # Remove patronymics
    name = re.sub(r"o['']g['']li|qizi", '', name, flags=re.IGNORECASE)
    # Remove special characters and extra spaces
    name = re.sub(r'[^\w\s]', '', name)
    # Convert to lowercase and strip
    name = name.lower().strip()
    # Remove extra spaces
    name = re.sub(r'\s+', ' ', name)
    return name

# Preprocess df1 and df2
df2['clean_name'] = df2['Full name'].apply(clean_name_for_comparison)

# For df1, combine last_name, first_name, and middle_name, then clean
df1['clean_name'] = df1.apply(
    lambda row: clean_name_for_comparison(f"{row['last_name']} {row['first_name']} {row['middle_name']}"), 
    axis=1
)
df1['words'] = df1['clean_name'].str.split()

# Create comparison names for each person in df2
comparison_results = []

# For each person in the smaller dataset (df2)
for _, row2 in df2.iterrows():
    clean_name2 = row2['clean_name']
    words2 = set(clean_name2.split())
    if not words2:
        continue  # Skip if no name left after cleaning
    
    first_word = clean_name2.split()[0]
    
    # Use regex to find exact word matches in last_name or first_name of df1
    regex_pattern = re.compile(r'\b{}\b'.format(re.escape(first_word)), re.IGNORECASE)
    
    # Check if first_word is in last_name or first_name (case-insensitive)
    mask_last = df1['last_name'].str.contains(regex_pattern, na=False)
    mask_first = df1['first_name'].str.contains(regex_pattern, na=False)
    potential_matches = df1[mask_last | mask_first]
    
    # Check only these potential matches
    for _, row1 in potential_matches.iterrows():
        words1 = set(row1['words'])
        common_words = words1.intersection(words2)
        if len(common_words) >= 2:
            comparison_results.append({
                'DB2_Name': row2['Full name'],
                'DB1_Name': f"{row1['last_name']} {row1['first_name']}",
                'DB2_Center': row2['Edu center'],
                'DB1_Center': row1['center_name'],
                'DB2_Course': row2['certificate'],
                'DB1_Course': row1['courses']
            })

# Convert results to DataFrame and save
results_df = pd.DataFrame(comparison_results)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
results_df.to_excel(f'name_matching_results_{timestamp}.xlsx', index=False)

print(f"Found {len(results_df)} potential matches")
print(f"Results saved to: name_matching_results_{timestamp}.xlsx")

if len(results_df) > 0:
    print("\nFirst few matches:")
    print(results_df[['DB2_Name', 'DB1_Name']].head())

Found 3448 potential matches
Results saved to: name_matching_results_20250122_153645.xlsx

First few matches:
                                            DB2_Name               DB1_Name
0                        Yerseitov Alixan Kanatovich       YERSEITOV ALIXAN
1                        Yerseitov Alixan Kanatovich       YERSEITOV ALIXAN
2                  Nurullayev Shohjahon Dilshodovich   NURULLAYEV SHOHJAHON
3  Turayev Xolrahmon Umorbek o'g'li (Allamov Umor...      TURAYEV XOLRAHMON
4               Jumanazarov Dostonbek Sherzod O‘g‘li  JUMANAZAROV JAVLONBEK


In [ ]:
import pandas as pd
import re
from datetime import datetime

# Read both Excel files
df1 = pd.read_excel('graduates.xlsx')
df2 = pd.read_excel('table.xlsx')

# Clean data
df1 = df1.fillna('')
df2 = df2.fillna('')

def clean_name_for_comparison(name):
    # Remove parentheses and their contents
    name = re.sub(r'\([^)]*\)', '', name)
    # Remove patronymics
    name = re.sub(r"o['']g['']li|qizi", '', name)
    # Remove special characters and extra spaces
    name = re.sub(r'[^\w\s]', '', name)
    # Convert to lowercase and strip
    name = name.lower().strip()
    # Remove extra spaces
    name = re.sub(r'\s+', ' ', name)
    return name

# Process only the smaller dataset first
df2['clean_name'] = df2['Full name'].apply(clean_name_for_comparison)

# Create comparison names for each person in df2
comparison_results = []

# For each person in the smaller dataset
for _, row2 in df2.iterrows():
    clean_name2 = row2['clean_name']
    words2 = set(clean_name2.split())
    
    # Create a filter for df1 based on the first word of the name
    first_word = clean_name2.split()[0]
    potential_matches = df1[
        (df1['last_name'].str.lower().str.contains(first_word, na=False)) |
        (df1['first_name'].str.lower().str.contains(first_word, na=False))
    ]
    
    # Check only these potential matches
    for _, row1 in potential_matches.iterrows():
        clean_name1 = clean_name_for_comparison(f"{row1['last_name']} {row1['first_name']}")
        words1 = set(clean_name1.split())
        
        if len(words1.intersection(words2)) >= 2:
            comparison_results.append({
                'DB2_Name': row2['Full name'],
                'DB1_Name': f"{row1['last_name']} {row1['first_name']}",
                'DB2_Center': row2['Edu center'],
                'DB1_Center': row1['center_name'],
                'DB2_Course': row2['certificate'],
                'DB1_Course': row1['courses']
            })

# Convert results to DataFrame
results_df = pd.DataFrame(comparison_results)

# Save results
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
results_df.to_excel(f'name_matching_results_{timestamp}.xlsx', index=False)

print(f"Found {len(results_df)} potential matches")
print(f"Results saved to: name_matching_results_{timestamp}.xlsx")

# Print first few matches for quick review
if len(results_df) > 0:
    print("\nFirst few matches:")
    print(results_df[['DB2_Name', 'DB1_Name']].head())

Found 227 potential matches
Results saved to: name_matching_results_20250122_151938.xlsx

First few matches:
                                            DB2_Name               DB1_Name
0                        Yerseitov Alixan Kanatovich       YERSEITOV ALIXAN
1                        Yerseitov Alixan Kanatovich       YERSEITOV ALIXAN
2                  Nurullayev Shohjahon Dilshodovich   NURULLAYEV SHOHJAHON
3  Turayev Xolrahmon Umorbek o'g'li (Allamov Umor...      TURAYEV XOLRAHMON
4             Yaqubboyev Jamoladdin Shoxnazar O‘g‘li  YAQUBBOYEV JAMOLADDIN


In [ ]:
df1 = pd.read_excel('graduates.xlsx')
df1

In [ ]:
df2 = pd.read_excel('table.xlsx')
df2